# LLM Evaluation on Greek Protipa Exams

In [ ]:
import json
import logging
import requests
import os
import sys
import random
import time
import traceback
from pathlib import Path

import lm_eval
from lm_eval.utils import make_table
import pandas as pd
import yaml
from datasets import load_dataset
from datasets import load_dataset, concatenate_datasets
from dotenv import load_dotenv, find_dotenv
from lm_eval.models.openai_completions import OpenAIChatCompletion
from lm_eval.tasks import ConfigurableTask, TaskManager

import IPython.display

# Setup Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

## Environment Setup

In [ ]:
load_dotenv(find_dotenv())

api_base = os.getenv("OPENAI_BASE_URL")

results_dir = Path(os.getenv("RESULTS_DIR", "tmp"))
results_dir.mkdir(parents=True, exist_ok=True)

models_to_test = ["gemma3-27b-it", "krikri-dpo-context"]
logger.info(f"Target models: {models_to_test}")

In [ ]:
project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    logger.info(f"✅ Added {src_path} to sys.path!")

try:
    from protipa_exams_dataset.data_loader import load_protipa_dataset, apply_matching_processing
    logger.info("🚀 Success! data_loader imported from src.")
except ImportError as e:
    logger.warning(f"⚠️ Could not find module/function. Please check the structure in src. Error: {e}")

Διαθέσιμα μοντέλα

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")
api_base = os.getenv("OPENAI_BASE_URL")

if api_base.endswith("/chat/completions"):
    api_base = api_base.replace("/chat/completions", "")
if not api_base.endswith("/v1"):
    api_base = api_base.rstrip("/") + "/v1"

try:
    response = requests.get(
        f"{api_base}/models", 
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=10
    )
    
    if response.status_code == 200:
        data = response.json()
        models = data.get('data', [])
        
        targets = ['llama', 'mistral', 'gemma', 'gpt']
        found_models = []
        
        for m in models:
            mid = m['id']
            if any(t in mid.lower() for t in targets):
                found_models.append(mid)
        
        print(json.dumps(found_models, indent=4))
        
    else:
        print(f"Error: {response.text}")

except Exception as e:
    print(f"Connection Error: {e}")

In [ ]:
#Saving results from closed tasks

file_path = "../results/closed_aggregate_test/llama-krikri-8b-instruct-v1.5/closed_aggregate_results.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/closed_aggregate_test/llama-krikri-8b-instruct-v1.5/krikri_closed_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

In [ ]:
#Saving results from closed tasks_few_shot_final

file_path = "../results/closed_aggregate_final/ilsp__Llama-Krikri-8B-Instruct-v1.5/closed_aggregate_final.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/closed_aggregate_final/ilsp__Llama-Krikri-8B-Instruct-v1.5/krikri_closed_final_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

In [ ]:
#Saving results from open tasks

file_path = "../results/open_aggregate_test/llama-krikri-8b-instruct-v1.5/open_aggregate_results.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/open_aggregate_test/llama-krikri-8b-instruct-v1.5/krikri_open_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

In [ ]:
#Saving results from open tasks_private_few_shot

file_path = "../results/open_aggregate_private/llama-krikri-8b-instruct-v1.5/open_aggregate_private_results.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/open_aggregate_private/llama-krikri-8b-instruct-v1.5/krikri_open_private_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

In [ ]:
#Saving results from structured tasks

file_path = "../results/structured_aggregate_test/llama-krikri-8b-instruct-v1.5/structured_aggregate_results.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/structured_aggregate_test/llama-krikri-8b-instruct-v1.5/krikri_structured_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

In [ ]:
#Saving results from structured tasks_few_shot_final

file_path = "../results/structured_aggregate_final/ilsp__Llama-Krikri-8B-Instruct-v1.5/structured_aggregate_final.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/structured_aggregate_final/ilsp__Llama-Krikri-8B-Instruct-v1.5/krikri_structured_results_final_few_shot.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))